# MMUbiPred–Context complementarity diagnostic v1

This CPU-only notebook analyzes frozen, already-saved predictions. It performs no training or model inference. Only corrected development out-of-fold evidence controls whether gated fusion is worth screening; optional historical and external sources are descriptive and cannot change that recommendation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
import json, numpy, scipy
print('CPU-only analysis; a GPU runtime is not required.')
print('NumPy:', numpy.__version__, 'SciPy:', scipy.__version__)

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

In [ ]:
DEVELOPMENT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-v1')
OUTPUT_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/analysis/mmubipred-context-complementarity-v1')
OUTPUT_PATH = OUTPUT_ROOT / 'complementarity.json'

required = [
    DEVELOPMENT_RUN_DIR / 'stabilized_summary.json',
    DEVELOPMENT_RUN_DIR / 'stabilized_oof_predictions.npz',
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Copy these two files from the Drive account that completed notebook 09 into ' + str(DEVELOPMENT_RUN_DIR) + ': ' + ', '.join(path.name for path in missing)
    )
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Required development artifacts found.')
print('Persistent output:', OUTPUT_PATH)

## Locate optional saved evaluations

Missing optional sources are normal. They will be skipped. They are descriptive only and cannot influence the fusion recommendation.

In [ ]:
optional_candidates = [
    ('historical_matched_test', Path('/content/drive/MyDrive/MMUbiPred-research/experiments/exact-mmubipred-context-equal-ensemble-exploratory-v1/ensemble_test_predictions.npz')),
    ('external_dbptm_ptmgpt2_v3', Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v3/evaluation/external_test_predictions.npz')),
    ('external_dbptm_disjoint_holdout_v2', Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-disjoint-holdout-v2/evaluation/external_test_predictions.npz')),
]
available_optional_sources = [(name, path) for name, path in optional_candidates if path.exists()]
display({
    'available': [(name, str(path)) for name, path in available_optional_sources],
    'missing_and_skipped': [name for name, path in optional_candidates if not path.exists()],
})

In [ ]:
if not OUTPUT_PATH.exists():
    command = [
        sys.executable, '-u', 'experiment/scripts/analyze_prediction_complementarity.py',
        '--development-run-dir', DEVELOPMENT_RUN_DIR,
        '--output', OUTPUT_PATH,
    ]
    for name, path in available_optional_sources:
        command.extend(['--saved-prediction-source', name, path])
    run_live(command)
else:
    print('Existing completed analysis found; saved predictions were not reanalyzed.')

result = json.loads(OUTPUT_PATH.read_text())

In [ ]:
local_context = result['development']['local_vs_context']
context_stack = result['development']['context_vs_crossfit_residual_stack']
display({
    'local_metrics': local_context['reference_metrics'],
    'context_metrics': local_context['candidate_metrics'],
    'local_vs_context_paired_correctness': local_context['paired_correctness'],
    'class_conditioned_paired_correctness': local_context['class_conditioned_paired_correctness'],
    'dependence': local_context['dependence'],
    'oracle_upper_bound': local_context['label_informed_oracle_upper_bound'],
})
display({
    'crossfit_stack_metrics': context_stack['candidate_metrics'],
    'stack_minus_context': context_stack['candidate_minus_reference'],
    'fusion_diagnostic': result['fusion_diagnostic'],
})

In [ ]:
display([
    {
        'fold': fold['fold'],
        'context_minus_local_mcc': fold['candidate_minus_reference']['mcc'],
        'local_only_correct': fold['paired_correctness']['reference_only_correct'],
        'context_only_correct': fold['paired_correctness']['candidate_only_correct'],
        'both_wrong': fold['paired_correctness']['both_wrong'],
    }
    for fold in result['development']['local_vs_context_by_fold']
])
display([
    {
        'source': source['name'],
        'context_minus_paper': source['paper_vs_context']['candidate_minus_reference'],
        'paired_correctness': source['paper_vs_context']['paired_correctness'],
        'may_change_recommendation': source['may_change_development_recommendation'],
    }
    for source in result['optional_post_evaluation_sources']
])

## Stop and report

Send the three displayed outputs. Do not initiate a gated-fusion training run unless the development diagnostic explicitly supports a separately predeclared screen.